# Tutor Scheduling with OR-Tools

This notebook builds a small operations research project in Python using **Google OR-Tools CP-SAT**.

## Goal
Assign tutors to sessions while:
- respecting availability,
- covering as many sessions as possible,
- balancing workload,
- rewarding preferred assignments.

## Why this matters
This is a realistic scheduling problem similar to workforce scheduling examples in OR-Tools.

In [13]:
!pwd
!ls

/content
availability.csv    sample_data   summary.csv	   tutors.csv
final_schedule.csv  sessions.csv  tutor_loads.csv


## Import libraries

We use:
- `pandas` for working with CSV files,
- `cp_model` from OR-Tools to build and solve the optimization model.

## Load the CSV files

These files contain:
- tutor workload limits,
- session information,
- tutor availability and preferences.

If your files are in the current Colab folder, use the filenames directly.
If they are in Google Drive, replace the paths with the full Drive path.

In [14]:
import pandas as pd

tutors_df = pd.read_csv("tutors.csv")
sessions_df = pd.read_csv("sessions.csv")
availability_df = pd.read_csv("availability.csv")

display(tutors_df)
display(sessions_df)
display(availability_df)

,tutor,max_shifts,min_shifts
0,Alex,3,1
1,Bri,3,1
2,Chris,2,1
3,Dani,2,1
4,Evan,2,0


,session_id,day,time_slot,required_tutors
0,Mon_9,Mon,9AM,1
1,Mon_11,Mon,11AM,1
2,Tue_9,Tue,9AM,1
3,Tue_1,Tue,1PM,1
4,Wed_11,Wed,11AM,1
5,Thu_1,Thu,1PM,1


,tutor,session_id,available,preferred
0,Alex,Mon_9,1,1
1,Alex,Mon_11,1,0
2,Alex,Tue_9,1,1
3,Alex,Tue_1,0,0
4,Alex,Wed_11,1,1
5,Alex,Thu_1,0,0
6,Bri,Mon_9,1,0
7,Bri,Mon_11,1,1
8,Bri,Tue_9,0,0
9,Bri,Tue_1,1,1


## Convert the tables into Python objects

OR-Tools works best when we convert the DataFrames into:
- lists of tutors and sessions,
- dictionaries for limits and requirements,
- dictionaries for availability and preferences.

In [17]:
tutors = tutors_df["tutor"].tolist()
sessions = sessions_df["session_id"].tolist()

max_shifts = dict(zip(tutors_df["tutor"], tutors_df["max_shifts"]))
min_shifts = dict(zip(tutors_df["tutor"], tutors_df["min_shifts"]))

required_tutors = dict(zip(sessions_df["session_id"], sessions_df["required_tutors"]))
session_day = dict(zip(sessions_df["session_id"], sessions_df["day"]))

availability = {}
preferences = {}

for _, row in availability_df.iterrows():
  key = (row["tutor"], row["session_id"])
  availability[key] = int(row["available"])
  preferences[key] = int(row["preferred"])

print("Tutors:", tutors)
print("Sesions:", sessions)

Tutors: ['Alex', 'Bri', 'Chris', 'Dani', 'Evan']
Sesions: ['Mon_9', 'Mon_11', 'Tue_9', 'Tue_1', 'Wed_11', 'Thu_1']


## Create the optimization model

A CP-SAT model stores:
- decision variables,
- constraints,
- the objective function.

In [18]:
model = cp_model.CpModel()

## Create decision variables

We define a binary variable:

`x[(t, s)] = 1` if tutor `t` is assigned to session `s`, otherwise `0`.

These variables represent the decisions the solver will make.

In [19]:
x = {}

for t in tutors:
  for s in sessions:
    x[(t, s)] = model.NewBoolVar(f"assign_{t}_{s}")

## Add availability constraints

If a tutor is not available for a session, the solver must assign 0 for that tutor-session pair.

In [21]:
for t in tutors:
  for s in sessions:
    if availability.get((t, s), 0) == 0:
      model.Add(x[(t, s)] == 0)

## Add coverage constraints

Each session needs a required number of tutors.

To keep the model flexible, we add an `uncovered` variable for each session.
This lets the model solve even if perfect coverage is impossible.

In [22]:
uncovered = {}

for s in sessions:
  uncovered[s] = model.NewIntVar(0, required_tutors[s], f"uncovered_{s}")

  model.Add(
      sum(x[(t, s)] for t in tutors) + uncovered[s] == required_tutors[s]
  )

## Add workload constraints

Each tutor must work:
- at least their minimum number of shifts,
- at most their maximum number of shifts.

In [23]:
for t in tutors:
  total_load = sum(x[(t, s)] for s in sessions)
  model.Add(total_load >= min_shifts[t])
  model.Add(total_load <= max_shifts[t])

## Add a simple scheduling rule

For now, each tutor can work at most one session per day.

Later, this can be replaced with a more realistic time-overlap constraint using start and end times.

In [24]:
days = sessions_df["day"].unique()

for t in tutors:
  for day in days:
    day_sessions = [s for s in sessions if session_day[s] == day]
    model.Add(sum(x[(t, s)] for s in day_sessions) <= 1)

## Measure fairness

To balance workload, we create:
- `load[t]` = number of sessions assigned to tutor `t`,
- `max_load` = largest workload,
- `min_load` = smallest workload,
- `fairness_gap` = difference between them.

A smaller fairness gap means a more balanced schedule.

In [25]:
load = {}

for t in tutors:
  load[t] = model.NewIntVar(0, len(sessions), f"load_{t}")
  model.Add(load[t] == sum(x[(t, s)] for s in sessions))

max_load = model.NewIntVar(0, len(sessions), "max_load")
min_load = model.NewIntVar(0, len(sessions), "min_load")

model.AddMaxEquality(max_load, [load[t] for t in tutors])
model.AddMinEquality(min_load, [load[t] for t in tutors])

fairness_gap = model.NewIntVar(0, len(sessions), "fairness_gap")
model.Add(fairness_gap == max_load - min_load)

## Define the objective

We want to:
1. minimize uncovered sessions,
2. minimize imbalance,
3. reward preferred assignments.

Since OR-Tools solves one objective expression, we combine them using weights.

In [27]:
total_uncovered = sum(uncovered[s] for s in sessions)

preference_score = sum(
    preferences.get((t, s), 0) * x[(t, s)]
    for t in tutors
    for s in sessions
)

model.Minimize(100 * total_uncovered + 5 * fairness_gap - 2 * preference_score)

## Solve the model

Now we ask OR-Tools to search for a feasible or optimal schedule.

In [28]:
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 10

status = solver.Solve(model)

## View the results

If a solution is found, we print:
- session assignments,
- coverage and fairness statistics,
- a final DataFrame for easy viewing.

In [31]:
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
  print("No feasible solution found")
else:
  print("Solution found!")
  print()

  assignment_rows = []

  for s in sessions:
    assigned = []
    for t in tutors:
      if solver.Value(x[(t, s)]) == 1:
        assigned.append(t)
        assignment_rows.append({
            "session_id": s,
            "tutor": t,
            "day": session_day[s],
            "preferred_assignment": preferences.get((t, s), 0)})
    if assigned:
      print(f"{s}: {assigned}")
    else:
      print(f"{s}: UNFILED")


  print()
  print("Total uncovered sessions:", sum(solver.Value(uncovered[s]) for s in sessions))
  print("Preference matches:", sum(preferences.get((t, s), 0) * solver.Value(x[(t, s)]) for t in tutors for s in sessions))
  print("Fairness gap:", solver.Value(fairness_gap))
  print("Objective value:", solver.ObjectiveValue())

  schedule_df = pd.DataFrame(assignment_rows)
  display(schedule_df)

Solution found!

Mon_9: ['Dani']
Mon_11: ['Bri']
Tue_9: ['Dani']
Tue_1: ['Evan']
Wed_11: ['Alex']
Thu_1: ['Chris']

Total uncovered sessions: 0
Preference matches: 6
Fairness gap: 1
Objective value: -7.0


,session_id,tutor,day,preferred_assignment
0,Mon_9,Dani,Mon,1
1,Mon_11,Bri,Mon,1
2,Tue_9,Dani,Tue,1
3,Tue_1,Evan,Tue,1
4,Wed_11,Alex,Wed,1
5,Thu_1,Chris,Thu,1


## Reflection questions

After running the notebook, try changing one thing at a time:

- What happens if one tutor becomes unavailable?
- What happens if a session requires 2 tutors instead of 1?
- What happens if the fairness penalty increases from 5 to 20?
- What happens if preferences are removed from the objective?

This is one of the best ways to learn optimization modeling by doing.